#Generation d'une simulation de combat

In [ ]:
import numpy as np
import random

# matplotlib est requis pour les graphiques (déjà préinstallé sur Google Colab)
!pip install -q matplotlib
import matplotlib.pyplot as plt

# NOTE : faker n'est plus utilisé (les prénoms viennent de STATS_REGION),
# l'installation et l'import ont été retirés pour accélérer le démarrage.

## Création des humains

In [ ]:
import numpy as np
import random
import itertools

class Human:
    _counter = 0
    STATS_REGION = {
        'Europe_Amerique_Nord': {'locales': ['fr_FR', 'en_US', 'de_DE', 'en_GB'], 'taille_h': 178, 'taille_f': 164, 'imc_moyen': 26.5, 'eco_factor': 45000, 'prenoms_h': ['Jean', 'Pierre', 'Luc', 'John', 'Michael'], 'prenoms_f': ['Marie', 'Sophie', 'Claire', 'Emma', 'Sarah']},
        'Asie_Est': {'locales': ['ja_JP', 'ko_KR', 'zh_CN'], 'taille_h': 172, 'taille_f': 159, 'imc_moyen': 23.0, 'eco_factor': 25000, 'prenoms_h': ['Takeshi', 'Kenji', 'Wei'], 'prenoms_f': ['Sakura', 'Yui', 'Mei']},
        'Asie_Sud': {'locales': ['hi_IN', 'en_IN'], 'taille_h': 165, 'taille_f': 152, 'imc_moyen': 22.0, 'eco_factor': 6000, 'prenoms_h': ['Raj', 'Amit', 'Arjun'], 'prenoms_f': ['Priya', 'Ananya', 'Deepika']},
        'Afrique': {'locales': ['fr_MA', 'en_NG', 'sw_KE'], 'taille_h': 170, 'taille_f': 158, 'imc_moyen': 23.5, 'eco_factor': 5000, 'prenoms_h': ['Mohammed', 'Amadou', 'Kofi'], 'prenoms_f': ['Fatou', 'Amina', 'Ngozi']},
        'Amerique_Latine': {'locales': ['es_MX', 'es_AR', 'pt_BR'], 'taille_h': 172, 'taille_f': 160, 'imc_moyen': 27.0, 'eco_factor': 15000, 'prenoms_h': ['Carlos', 'Jose', 'Luis'], 'prenoms_f': ['Maria', 'Sofia', 'Isabella']},
        'Oceanie': {'locales': ['en_AU', 'en_NZ'], 'taille_h': 176, 'taille_f': 163, 'imc_moyen': 27.5, 'eco_factor': 42000, 'prenoms_h': ['Jack', 'Liam', 'Oliver'], 'prenoms_f': ['Charlotte', 'Olivia', 'Mia']}
    }
    CLASSES_SOCIALES = ['Indigent', 'Misérable', 'Pauvre', 'Modeste', 'Classe moyenne inf.', 'Classe moyenne sup.', 'Aisé', 'Riche', 'Très riche', 'Fortune']

    # OPTIMISATION : la liste des clés de région et les poids sont calculés UNE FOIS
    # au niveau de la classe, au lieu d'être reconstruits à chaque Human().
    REGION_KEYS = list(STATS_REGION.keys())
    # BUGFIX : les poids sommaient à 1.05 (au lieu de 1.0). Normalisés pour représenter
    # de vraies probabilités (Asie_Est ramené de 0.30 à 0.25).
    REGION_WEIGHTS = [0.155, 0.225, 0.285, 0.235, 0.095, 0.005]  # BUGFIX : somme = 1.0 (Oceanie 0.05 -> 0.10)
    # OPTIMISATION : poids cumulés précalculés -> random.choices(cum_weights=...) évite
    # de re-sommer la distribution à chaque tirage (8192 appels à la génération).
    REGION_CUM_WEIGHTS = list(itertools.accumulate(REGION_WEIGHTS))

    # OPTIMISATION : tranches d'âge et poids cumulés hissés en constantes de classe.
    # Avant, deux listes littérales étaient ré-allouées à chaque _calculer_age().
    AGE_BUCKETS = ['enfant', 'adulte', 'mature', 'vieux']
    AGE_CUM_WEIGHTS = list(itertools.accumulate([0.30, 0.35, 0.25, 0.10]))

    # OPTIMISATION (#5) : proportions de PV par membre (constante de classe, partagee
    # par _construire_corps -> _initialiser_corps et integrer_joueur).
    PROPORTIONS_CORPS = {
        'tete': 0.8, 'torse': 1.0, 'bras_g': 0.4, 'bras_d': 0.4, 'jambe_g': 0.5, 'jambe_d': 0.5
    }

    def __init__(self):
        Human._counter += 1
        self.id = Human._counter
        self.region_key = random.choices(Human.REGION_KEYS, cum_weights=Human.REGION_CUM_WEIGHTS, k=1)[0]
        region_data = self.STATS_REGION[self.region_key]

        # OPTIMISATION : random.random() au lieu de np.random.choice (beaucoup plus
        # rapide pour un tirage binaire scalaire), distribution identique.
        self.sexe = 'Homme' if random.random() < 0.505 else 'Femme'
        self._generer_identite(region_data)
        self._calculer_age()
        self._calculer_physique(region_data)
        self._calculer_sante()

        # --- AJOUT ICI ---
        self._calculer_mental()

        self._calculer_stats_combat()
        self._calculer_entrainement()
        self._calculer_richesse(region_data)
        self._initialiser_corps()

        # BUGFIX : l'incapacité liée au grand âge est tirée UNE SEULE FOIS à la création.
        # Avant, elle était re-tirée à chaque appel de recuperation(), ce qui rendait
        # la capacité de combat aléatoire d'un round à l'autre.
        self._handicap_grand_age = (self.age > 85 and random.random() < 0.8)
        self._verifier_capacite_combat()

    def _generer_identite(self, region_data):
        if self.sexe == 'Homme': self.prenom = random.choice(region_data['prenoms_h'])
        else: self.prenom = random.choice(region_data['prenoms_f'])

    def _calculer_age(self):
        choix = random.choices(Human.AGE_BUCKETS, cum_weights=Human.AGE_CUM_WEIGHTS, k=1)[0]
        if choix == 'enfant': self.age = random.randint(0, 17)
        elif choix == 'adulte': self.age = random.randint(18, 40)
        # 3.2 : pyramide realiste -> densite decroissante avec l'age (la mortalite
        # trie). triangular(min, max, mode=min) : bien plus de quadras que de
        # sexagenaires, et bien plus de septuagenaires que de centenaires.
        elif choix == 'mature': self.age = int(random.triangular(41, 66, 41))
        else: self.age = int(random.triangular(66, 101, 66))

    def _calculer_physique(self, region_data):
        base_taille = region_data['taille_h'] if self.sexe == 'Homme' else region_data['taille_f']
        # BUGFIX : suppression d'une ligne morte (facteur_croissance était calculé puis
        # systématiquement écrasé pour age<18, et valait toujours 1.0 pour age>=18).
        if self.age < 18: facteur_croissance = 0.5 + (self.age / 18 * 0.5)
        else: facteur_croissance = 1.0
        self.taille = base_taille * facteur_croissance + random.gauss(0, 3)
        imc_target = region_data['imc_moyen']
        if self.age < 18: imc_target = 16 + (self.age * 0.4)
        elif self.age > 60: imc_target *= 1.05
        self.poids = round(random.gauss(imc_target, 2) * (self.taille/100)**2, 1)

    def _calculer_sante(self):
        # 3.4 : courbe acceleree, prevalence realiste (jeunes ~2%, tres ages ~50%).
        proba_maladie = 0.02 + (self.age / 100) ** 2 * 0.5
        self.est_malade = random.random() < proba_maladie
        self.severite_maladie = min(1.0, random.expovariate(1 / 0.3)) if self.est_malade else 0.0

    # --- NOUVELLE METHODE ---
    def _calculer_mental(self):
        base = random.gauss(0.85, 0.15) # La plupart sont stables
        if self.age < 12: base -= 0.2
        if self.age > 80: base -= 0.1
        if self.est_malade: base -= self.severite_maladie * 0.3
        self.mental = max(0.1, min(1.0, base))
        self.mental_max = self.mental
        # 5.1 : resilience psychologique individuelle (encaisse plus ou moins le
        # trauma). Variabilite via betavariate, attenuee aux ages extremes.
        res = random.betavariate(2, 2)
        if self.age < 14: res *= 0.7
        if self.age > 75: res *= 0.85
        self.resilience = max(0.05, min(1.0, res))

    def _facteur_maladie(self):
        # BUGFIX #3 : malus multiplicatif REVERSIBLE lie a la maladie. Vaut 1.0 une
        # fois gueri, ce qui permet a recuperation() de restaurer la pleine stat.
        return (1 - self.severite_maladie * 0.5) if self.est_malade else 1.0

    def _calculer_stats_combat(self):
        # BUGFIX : force bornee a >= 1. Une force negative (tirage gaussien bas +
        # faible poids) produisait des degats negatifs qui SOIGNAIENT la cible.
        # OPTIMISATION (#1) : random.gauss au lieu de np.random.normal (tirage scalaire
        # ~10x plus rapide, distribution identique) sur toute la generation.
        # 3.3 : seule la masse "utile" (maigre) fait la force. Au-dela d'un IMC
        # athletique (~25), le poids en plus est surtout du gras et ne compte qu'a
        # 20% -> un obese n'a pas une force de colosse.
        imc = self.poids / (self.taille / 100) ** 2
        poids_seuil = 25 * (self.taille / 100) ** 2
        masse_utile = self.poids if imc <= 25 else poids_seuil + (self.poids - poids_seuil) * 0.2
        self.force = max(1.0, random.gauss(30, 10) + masse_utile * 0.4)
        facteur_age_vitesse = 1.0 if self.age < 25 else max(0.5, 1 - (self.age - 25) * 0.01)
        self.vitesse = random.gauss(50, 8) * facteur_age_vitesse
        self.intelligence = random.gauss(100, 15) + max(0, (self.age - 20) * 0.2)

        # BUGFIX #3 : les stats max representent le potentiel SAIN. La maladie est un
        # malus separe (cf. _facteur_maladie) applique sur la valeur courante, afin
        # que la guerison de la maladie pendant recuperation() rende la force/vitesse.
        self.force_max = self.force
        self.vitesse_max = self.vitesse
        facteur_maladie = self._facteur_maladie()
        self.force *= facteur_maladie
        self.vitesse *= facteur_maladie

    def _calculer_entrainement(self):
        self.niveau_entrainement = 0.0
        if 16 < self.age < 50:
            base_proba = 0.35 if self.sexe == 'Homme' else 0.20
            if random.random() < base_proba:
                self.niveau_entrainement = random.betavariate(2, 5)  # OPTIMISATION #1
                self.resilience = min(1.0, self.resilience + self.niveau_entrainement * 0.3)  # 5.1 : l'entrainement endurcit
                # BUGFIX #3 : le bonus s'ajoute au potentiel SAIN (force_max/vitesse_max),
                # puis on reapplique le malus de maladie sur la valeur courante.
                self.force_max += self.niveau_entrainement * 20
                self.vitesse_max += self.niveau_entrainement * 10
                facteur_maladie = self._facteur_maladie()
                self.force = self.force_max * facteur_maladie
                self.vitesse = self.vitesse_max * facteur_maladie

    def _calculer_richesse(self, region_data):
        base_region = region_data['eco_factor']
        if self.age < 18: facteur_age = 0.05
        elif self.age < 25: facteur_age = 0.2 + (self.age - 18) * 0.05
        elif self.age < 60: facteur_age = 0.6 + min(0.4, (self.age - 25) * 0.01)
        else: facteur_age = max(0.4, 1.0 - (self.age - 60) * 0.02)
        wealth = base_region * facteur_age * random.lognormvariate(0, 0.8)
        self.richesse = round(wealth, 2)
        if self.richesse < base_region * 0.1: index = 0
        elif self.richesse < base_region * 0.3: index = 1
        elif self.richesse < base_region * 0.5: index = 2
        elif self.richesse < base_region * 0.8: index = 3
        elif self.richesse < base_region * 1.2: index = 4
        elif self.richesse < base_region * 2.0: index = 5
        elif self.richesse < base_region * 4.0: index = 6
        elif self.richesse < base_region * 10.0: index = 7
        elif self.richesse < base_region * 50.0: index = 8
        else: index = 9
        self.classe_sociale = self.CLASSES_SOCIALES[index]

    # OPTIMISATION (#5) : construction du corps factorisee a partir de PROPORTIONS_CORPS
    # (plus de 6 ratios reecrits ; partage avec integrer_joueur).
    @staticmethod
    def _construire_corps(poids):
        base_pv = 20 + (poids / 2)
        return {membre: {'pv_max': round(base_pv * prop), 'pv_actuels': round(base_pv * prop)}
                for membre, prop in Human.PROPORTIONS_CORPS.items()}

    def _initialiser_corps(self):
        self.corps = Human._construire_corps(self.poids)
        self.pv_total = sum(m['pv_actuels'] for m in self.corps.values())

    def _verifier_capacite_combat(self):
        capable = True
        if self.age < 8: capable = False
        if getattr(self, '_handicap_grand_age', False): capable = False
        if self.severite_maladie > 0.8: capable = False
        # BUGFIX : un combattant dont la tete ou le torse est detruit ne peut plus
        # se battre. Avant, _verifier_capacite_combat ignorait les PV du corps : un
        # "survivant" a la tete a -50 (issu de la branche double-incapacite de
        # combat_a_mort) restait capable et pouvait avancer dans le tournoi sans
        # jamais avoir combattu reellement.
        corps = getattr(self, 'corps', None)
        if corps is not None and (corps['tete']['pv_actuels'] <= 0 or corps['torse']['pv_actuels'] <= -20):
            capable = False
        self.peut_se_battre = capable

    def __str__(self):
        return (f"ID:{self.id:03d} {self.prenom:10s} ({self.age}ans, {self.sexe[0]}) | For:{self.force:.0f} Vit:{self.vitesse:.0f} | {self.classe_sociale}")

## --- 2. FONCTIONS DE SIMULATION ---


In [ ]:
# Constantes de ciblage (definies une seule fois, pas a chaque coup).
import itertools
CIBLES = ['tete', 'torse', 'bras_g', 'bras_d', 'jambe_g', 'jambe_d']
POIDS_CIBLES_NORMAL = [0.1, 0.4, 0.1, 0.1, 0.15, 0.15]     # ciblage instinctif
POIDS_CIBLES_PANIQUE = [0.2, 0.3, 0.1, 0.1, 0.15, 0.15]    # mental bas -> coups disperses
# 2.2 : ciblage tactique d'un combattant lucide ET intelligent (vise les zones vitales).
POIDS_CIBLES_TACTIQUE = [0.18, 0.50, 0.06, 0.06, 0.10, 0.10]
CUM_CIBLES_NORMAL = list(itertools.accumulate(POIDS_CIBLES_NORMAL))
CUM_CIBLES_PANIQUE = list(itertools.accumulate(POIDS_CIBLES_PANIQUE))
CUM_CIBLES_TACTIQUE = list(itertools.accumulate(POIDS_CIBLES_TACTIQUE))

# 2.1 : echelle de degats. Les degats bruts (force ~50-90) ecrasaient des PV faibles
# -> un seul coup tuait. On reduit fortement pour des duels a plusieurs echanges ;
# la tete reste la zone la plus efficace, sans etre un one-shot.
DEGATS_SCALE = 0.22
MULT_ZONE = {'tete': 1.5, 'torse': 1.0}   # membres : 0.5 (valeur par defaut)

# 2.3 : cout d'endurance par action. L'entrainement economise le souffle,
# le surpoids et l'age le font fondre plus vite.
def _cout_endurance(c):
    imc = c.poids / (c.taille / 100) ** 2 if getattr(c, 'taille', 0) else 22.0
    cout = 0.030
    cout *= 1.0 - 0.4 * getattr(c, 'niveau_entrainement', 0.0)
    if imc > 27: cout *= 1.0 + (imc - 27) * 0.03
    if c.age > 45: cout *= 1.0 + (c.age - 45) * 0.01
    return cout


def combat_a_mort(combattant_1, combattant_2, verbose=True):
    # Verifications pre-combat (cas des incapables).
    if not combattant_1.peut_se_battre and not combattant_2.peut_se_battre:
        # Les DEUX sont incapables : on departe au total de PV, seul le perdant est
        # acheve, le survivant est stabilise au minimum vital puis reverifie.
        pv1 = sum(m['pv_actuels'] for m in combattant_1.corps.values())
        pv2 = sum(m['pv_actuels'] for m in combattant_2.corps.values())
        winner = combattant_1 if pv1 >= pv2 else combattant_2
        loser = combattant_2 if winner is combattant_1 else combattant_1
        loser.corps['tete']['pv_actuels'] = -50
        if winner.corps['tete']['pv_actuels'] <= 0:
            winner.corps['tete']['pv_actuels'] = 1
        if winner.corps['torse']['pv_actuels'] <= -20:
            winner.corps['torse']['pv_actuels'] = 1
        winner.pv_total = sum(m['pv_actuels'] for m in winner.corps.values())
        winner._verifier_capacite_combat()
        return winner
    if not combattant_1.peut_se_battre:
        combattant_1.corps['tete']['pv_actuels'] = -50
        return combattant_2
    if not combattant_2.peut_se_battre:
        combattant_2.corps['tete']['pv_actuels'] = -50
        return combattant_1

    if verbose:
        print(f"--- DUEL : {combattant_1.prenom} (M:{combattant_1.mental:.0%}) VS {combattant_2.prenom} (M:{combattant_2.mental:.0%}) ---")

    timer_c1, timer_c2 = 0.0, 0.0
    winner = None
    # 2.3 : endurance normalisee (1.0 = frais, 0.0 = epuise), indexee par is_c1.
    endurance = {True: 1.0, False: 1.0}

    # Garde-fou anti-boucle-infinie (double tetanie / impasse).
    tours = 0
    MAX_TOURS = 100000

    while True:
        tours += 1
        if tours > MAX_TOURS:
            pv1 = sum(m['pv_actuels'] for m in combattant_1.corps.values())
            pv2 = sum(m['pv_actuels'] for m in combattant_2.corps.values())
            winner = combattant_1 if pv1 >= pv2 else combattant_2
            if verbose: print("--- IMPASSE : combat departage aux points ---")
            break

        # Si les deux sont tetanises (mental <= 0.1), personne n'attaque -> on departage.
        if combattant_1.mental <= 0.1 and combattant_2.mental <= 0.1:
            pv1 = sum(m['pv_actuels'] for m in combattant_1.corps.values())
            pv2 = sum(m['pv_actuels'] for m in combattant_2.corps.values())
            winner = combattant_1 if pv1 >= pv2 else combattant_2
            if verbose: print("--- DOUBLE TETANIE : combat departage aux points ---")
            break

        if timer_c1 <= timer_c2:
            attaquant, defenseur = combattant_1, combattant_2
            timer_att = timer_c1
            is_c1 = True
        else:
            attaquant, defenseur = combattant_2, combattant_1
            timer_att = timer_c2
            is_c1 = False

        # Temps d'action de base (plus on est rapide, plus on agit souvent).
        base_time = 75.0 / attaquant.vitesse if attaquant.vitesse > 0 else 3.0
        if attaquant.mental < 0.3: base_time *= 1.5   # le stress ralentit

        # 2.3 : mise a jour de la fatigue de l'attaquant.
        end_att = endurance[is_c1]
        end_att = min(1.0, end_att + 0.006)                  # leger souffle entre deux actions
        end_att = max(0.0, end_att - _cout_endurance(attaquant))
        endurance[is_c1] = end_att
        facteur_fatigue = 0.4 + 0.6 * end_att                # 1.0 frais -> 0.4 epuise
        base_time *= 1.0 + (1.0 - end_att) * 0.6             # un epuise agit plus lentement

        if is_c1: timer_c1 += base_time
        else: timer_c2 += base_time
        temps_combat = timer_att

        # Crise de panique : mental brise -> passe son tour.
        if attaquant.mental <= 0.1:
            if verbose: print(f"[{temps_combat:.1f}s] {attaquant.prenom} est tetanise par la terreur !")
            continue

        # 2.2 : choix de la cible. Lucide + intelligent -> ciblage tactique vital ;
        # mental bas -> panique ; sinon instinctif.
        composure = attaquant.mental * min(1.0, attaquant.intelligence / 120.0)
        if composure >= 0.6:
            cum = CUM_CIBLES_TACTIQUE
        elif attaquant.mental < 0.5:
            cum = CUM_CIBLES_PANIQUE
        else:
            cum = CUM_CIBLES_NORMAL
        cible = random.choices(CIBLES, cum_weights=cum, k=1)[0]

        # 2.2 : la PRECISION depend de l'entrainement, des reflexes (vitesse) et du
        # sang-froid -- plus de l'intelligence (qui pilote desormais le ciblage).
        precision_base = 0.6 + 0.30 * attaquant.niveau_entrainement
        precision_base += (attaquant.vitesse - defenseur.vitesse) / 250
        precision_base *= (0.5 + 0.5 * attaquant.mental)     # malus si mental bas
        precision_base *= (0.6 + 0.4 * facteur_fatigue)      # 2.3 : un bras epuise vise mal
        precision_base = max(0.05, min(0.95, precision_base))

        if random.random() < precision_base:
            # 2.4 : DEFENSE ACTIVE. Le defenseur tente d'esquiver/parer selon ses
            # reflexes (vitesse), son entrainement, son sang-froid et son souffle.
            end_def = endurance[not is_c1]
            esquive = 0.10 + 0.25 * defenseur.niveau_entrainement
            esquive += (defenseur.vitesse - attaquant.vitesse) / 300
            esquive *= (0.4 + 0.6 * defenseur.mental)
            esquive *= (0.4 + 0.6 * (0.4 + 0.6 * end_def))
            esquive = max(0.0, min(0.75, esquive))
            if random.random() < esquive:
                endurance[not is_c1] = max(0.0, end_def - 0.02)   # esquiver coute de l'energie
                if verbose and random.random() < 0.2:
                    print(f"[{temps_combat:.1f}s] {defenseur.prenom} esquive l'attaque de {attaquant.prenom}.")
                continue

            # 2.1 : degats fortement reduits (DEGATS_SCALE) + modulation par la fatigue.
            degats = attaquant.force * random.uniform(0.8, 1.2) * DEGATS_SCALE
            degats *= MULT_ZONE.get(cible, 0.5)
            degats *= facteur_fatigue
            degats_finaux = max(1, round(degats))

            membre = defenseur.corps[cible]
            etait_deja_casse = membre['pv_actuels'] <= 0
            membre['pv_actuels'] -= degats_finaux

            if verbose:
                print(f"[{temps_combat:.1f}s] {attaquant.prenom} frappe {defenseur.prenom} au {cible} ({degats_finaux} dmg).")

            # Impact psychologique (5.1 : module par la resilience individuelle).
            res_def = getattr(defenseur, 'resilience', 0.5)
            defenseur.mental -= 0.02 * (degats_finaux / 10) * (1.3 - res_def)   # la douleur traumatise
            defenseur.mental = max(0, defenseur.mental)

            res_att = getattr(attaquant, 'resilience', 0.5)
            if attaquant.niveau_entrainement < 0.3:
                attaquant.mental -= 0.01 * (1.3 - res_att)     # les non-aguerris sont choques
            else:
                attaquant.mental -= 0.002 * (1.3 - res_att)    # les guerriers sont endurcis
            attaquant.mental = max(0, attaquant.mental)

            # Effets a la rupture du membre (transition >0 -> <=0).
            if membre['pv_actuels'] <= 0 and not etait_deja_casse:
                if 'jambe' in cible: defenseur.vitesse *= 0.5
                elif 'bras' in cible: defenseur.force *= 0.7
                defenseur.mental -= 0.15 * (1.3 - res_def)     # choc de la blessure grave
                defenseur.mental = max(0, defenseur.mental)
                defenseur._trauma_profond = True               # 4.3 : cicatrice psy durable
                if verbose: print(f"   >> {defenseur.prenom} est gravement blesse ! Panique...")

            # Mort physique.
            if defenseur.corps['tete']['pv_actuels'] <= 0 or defenseur.corps['torse']['pv_actuels'] <= -20:
                winner = attaquant
                winner.mental = max(0, winner.mental - 0.1 * (1.3 - res_att))
                winner._trauma_profond = True                  # 4.3 : tuer laisse une trace
                if verbose: print(f"--- MORT DE {defenseur.prenom} ---")
                break

            # Mort psychologique (capitulation).
            if defenseur.mental <= 0:
                if verbose: print(f"--- {defenseur.prenom} s'effondre mentalement et se laisse mourir ---")
                defenseur.corps['tete']['pv_actuels'] = 0
                winner = attaquant
                break

        else:
            if verbose and random.random() < 0.2:
                print(f"[{temps_combat:.1f}s] {attaquant.prenom} manque son coup.")

    # Post-combat.
    if winner:
        winner.pv_total = sum(m['pv_actuels'] for m in winner.corps.values())
        winner.mental = min(winner.mental_max, winner.mental + 0.05)   # leger apaisement
    return winner


In [36]:
# Taux de recuperation (PV/heure ou points/heure selon le cas).
TAUX_PHYS_NATUREL = 0.1
TAUX_PHYS_MEDICAL = 0.5
TAUX_MENTAL_NATUREL = 0.01
TAUX_MENTAL_MEDICAL = 0.05
# Maladie : etat reversible (taux de guerison par heure).
TAUX_MALADIE_NATUREL = 0.005
TAUX_MALADIE_MEDICAL = 0.02


def recuperation(individu, temps_heures, equipe_medecin):
    """
    Recuperation entre deux rounds. 24h de repos restaurent surtout la CONDITION
    physique et le moral ; les blessures profondes (membres detruits) ne se
    "reparent" pas en une journee.
    """
    taux_physique_reel = TAUX_PHYS_MEDICAL if equipe_medecin else TAUX_PHYS_NATUREL
    taux_mental_reel = TAUX_MENTAL_MEDICAL if equipe_medecin else TAUX_MENTAL_NATUREL

    # La douleur ralentit la recuperation mentale (ratio borne dans [0, 1]).
    pv_max_total = sum(m['pv_max'] for m in individu.corps.values())
    pv_actuel_total = sum(m['pv_actuels'] for m in individu.corps.values())
    ratio_sante = max(0.0, min(1.0, pv_actuel_total / pv_max_total)) if pv_max_total > 0 else 1.0

    # --- Guerison du corps ---
    pv_recup_total = temps_heures * taux_physique_reel
    for membre, data in individu.corps.items():
        pv_actuels = data['pv_actuels']
        pv_max = data['pv_max']
        if pv_actuels < pv_max:
            if pv_actuels <= 0:
                # 4.2 : fracture / membre detruit -> PAS de reparation en 24h. Avec une
                # equipe medicale on stabilise (anti-infection) et on amorce une
                # consolidation LENTE (~2 PV/jour) : le membre reste hors d'usage
                # plusieurs rounds avant de redevenir fonctionnel. Sans soins, quasi nul.
                if equipe_medecin:
                    data['pv_actuels'] = min(pv_max, pv_actuels + 2.0 * (temps_heures / 24.0))
                else:
                    data['pv_actuels'] = min(pv_max, pv_actuels + 0.2 * (temps_heures / 24.0))
            else:
                # 4.1 : blessure legere (PV>0). Le repos restaure surtout la CONDITION
                # (fatigue, contusions, coupures superficielles), pas une cicatrisation
                # lente en PV/h -> recuperation proportionnelle au deficit.
                deficit = pv_max - pv_actuels
                taux_condition = 0.6 if equipe_medecin else 0.30   # fraction du deficit par 24h
                data['pv_actuels'] = min(pv_max, pv_actuels + deficit * taux_condition * (temps_heures / 24.0))

    # --- 4.3 : Guerison mentale NON lineaire, liee a la resilience et au trauma ---
    resilience = getattr(individu, 'resilience', 0.5)
    max_mental = getattr(individu, 'mental_max', 1.0)
    # Cicatrice psychologique (PTSD) : un trauma profond abaisse durablement le
    # plafond mental -> la recuperation plafonne sous le niveau initial.
    if getattr(individu, '_trauma_profond', False):
        individu.mental_max = max(0.3, max_mental - 0.05 * (1 - resilience))
        max_mental = individu.mental_max
        individu._trauma_profond = False
    # Recuperation rapide loin du plafond, de plus en plus lente en l'approchant ;
    # modulee par la resilience individuelle et par la douleur (ratio_sante).
    ecart = max(0.0, max_mental - individu.mental)
    taux_mental_eff = taux_mental_reel * (0.5 + resilience)
    recup_mentale = temps_heures * taux_mental_eff * ratio_sante * (0.3 + 0.7 * ecart)
    individu.mental = min(max_mental, individu.mental + recup_mentale)

    # --- Guerison de la maladie (etat reversible) ---
    if getattr(individu, 'est_malade', False):
        taux_maladie = TAUX_MALADIE_MEDICAL if equipe_medecin else TAUX_MALADIE_NATUREL
        individu.severite_maladie = max(0.0, individu.severite_maladie - taux_maladie * temps_heures)
        if individu.severite_maladie <= 0.05:
            individu.est_malade = False
            individu.severite_maladie = 0.0

    # --- Recalcul des stats derivees (force/vitesse) selon l'etat des membres ---
    # On prend le MINIMUM de chaque paire (un membre hors d'usage penalise la paire).
    force_base = getattr(individu, 'force_max', individu.force)
    vitesse_base = getattr(individu, 'vitesse_max', individu.vitesse)
    etat_jambes = min(individu.corps['jambe_g']['pv_actuels'], individu.corps['jambe_d']['pv_actuels'])
    etat_bras = min(individu.corps['bras_g']['pv_actuels'], individu.corps['bras_d']['pv_actuels'])
    pv_max_jambe = individu.corps['jambe_g']['pv_max']
    pv_max_bras = individu.corps['bras_g']['pv_max']
    facteur_maladie = (1 - individu.severite_maladie * 0.5) if getattr(individu, 'est_malade', False) else 1.0

    if etat_jambes <= 0:
        individu.vitesse = vitesse_base * 0.1 * facteur_maladie   # quasi immobile
    else:
        ratio_jambes = min(1.0, etat_jambes / pv_max_jambe)
        individu.vitesse = vitesse_base * ratio_jambes * facteur_maladie

    if etat_bras <= 0:
        individu.force = force_base * 0.2 * facteur_maladie       # tres affaibli
    else:
        ratio_bras = min(1.0, etat_bras / pv_max_bras)
        individu.force = force_base * ratio_bras * facteur_maladie

    individu.pv_total = sum(m['pv_actuels'] for m in individu.corps.values())
    individu._verifier_capacite_combat()


## --- 3. FONCTION HUNGER GAMES (Logique du tournoi) ---

In [ ]:
def hunger_games(population, tracked_ids=None, equipe_medecin=False):
    # BUGFIX : equipe_medecin est desormais propage a recuperation(). Avant, le
    # tournoi appelait toujours recuperation(..., False), rendant toute la branche
    # medicale de recuperation() (taux_medical, stabilisation, platre) inatteignable.
    if tracked_ids is None:
        tracked_ids = []

    round_num = 1
    historique_joueurs = {pid: "En vie" for pid in tracked_ids}

    while len(population) > 1:
        print(f"\n=== ROUND {round_num} | Survivants : {len(population)} ===")
        survivors = []
        random.shuffle(population)

        for i in range(0, len(population), 2):
            if i + 1 >= len(population):
                # BUGFIX : le combattant exempté (population impaire) récupère aussi 24h.
                # Avant, il avançait au round suivant avec ses blessures non soignées,
                # désavantage non intentionnel par rapport aux vainqueurs de duel.
                recuperation(population[i], 24, equipe_medecin)
                survivors.append(population[i])
                break

            h1 = population[i]
            h2 = population[i+1]

            # COMBAT A MORT
            winner = combat_a_mort(h1, h2, verbose=False)

            # RECUPERATION : 24h (1 jour)
            recuperation(winner, 24, equipe_medecin)

            survivors.append(winner)

            # --- VERIFICATION DES JOUEURS ET AFFICHAGE COMPARATIF ---
            if winner == h1:
                # h2 est le perdant
                if h2.id in tracked_ids:
                    print(f"\n>>> LE JOUEUR (ID:{h2.id}) A ÉTÉ ÉLIMINÉ AU ROUND {round_num} <<<")
                    print(f"   PERDANT : {h2}") # Affiche les stats du joueur
                    print(f"   GAGNANT : {h1}") # Affiche les stats du tueur pour comparaison
                    historique_joueurs[h2.id] = f"Mort Round {round_num} par {h1.prenom}"
            else:
                # h1 est le perdant
                if h1.id in tracked_ids:
                    print(f"\n>>> LE JOUEUR (ID:{h1.id}) A ÉTÉ ÉLIMINÉ AU ROUND {round_num} <<<")
                    print(f"   PERDANT : {h1}") # Affiche les stats du joueur
                    print(f"   GAGNANT : {h2}") # Affiche les stats du tueur pour comparaison
                    historique_joueurs[h1.id] = f"Mort Round {round_num} par {h2.prenom}"

        population = survivors

        # --- Affichage Graphiques ---
        # BUGFIX : l'ancienne condition (== 20 / >= 30) n'etait jamais atteinte :
        # 8192 = 2^13 -> le tournoi dure 13 rounds. On affiche desormais le round 1
        # puis tous les 5 rounds (rounds reellement atteints).
        if round_num == 1 or round_num % 5 == 0:
            print(f"--- ANALYSE GRAPHIQUE DU ROUND {round_num} ---")
            # Décommentez les graphiques souhaités
            # graph_parite(population, round_num)
            # graph_force_vitesse(population, round_num)
            # graph_blessures(population, round_num)

        round_num += 1

    # --- BILAN FINAL ---
    champion = population[0]

    print(f"\n{'='*40}")
    print(f"LE VAINQUEUR EST : {champion}")
    print(f"{'='*40}")

    print(f"\n--- BILAN DES JOUEURS ---")
    for pid, status in historique_joueurs.items():
        if champion.id == pid:
            print(f"Joueur {pid} : VAINQUEUR DU TOURNOI !")
        else:
            print(f"Joueur {pid} : {status}")

    return champion

## 4 Génération des graphique

In [ ]:
def graph_parite(population, round_num):
    """
    Affiche un camembert de la répartition H/F.
    """
    hommes = sum(1 for h in population if h.sexe == 'Homme')
    femmes = len(population) - hommes

    labels = ['Hommes', 'Femmes']
    sizes = [hommes, femmes]
    colors = ['lightskyblue', 'lightcoral']

    plt.figure(figsize=(6, 6))
    plt.pie(sizes, labels=labels, colors=colors, autopct='%1.1f%%', startangle=90)
    plt.title(f"% d'hommes et de femmes au round {round_num}")
    plt.show()
    plt.close()  # OPTIMISATION : libère la figure (évite l'accumulation en mémoire si appelé en boucle)

In [ ]:
def graph_entrainement(population, round_num):
    """
    Affiche la distribution du niveau d'entraînement (skill) des survivants.
    """
    niveaux = [h.niveau_entrainement for h in population]

    plt.figure(figsize=(8, 5))
    plt.hist(niveaux, bins=20, color='green', alpha=0.7, edgecolor='black')
    plt.xlabel("Niveau d'entraînement (0 = Débutant, 1 = Expert)")
    plt.ylabel("Nombre d'individus")
    plt.title(f"Compétence de combat au round {round_num}")
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    plt.show()
    plt.close()  # OPTIMISATION : libère la figure

In [ ]:
def graph_age(population, round_num):
    """
    Affiche un histogramme des âges.
    """
    ages = [h.age for h in population]

    plt.figure(figsize=(8, 5))
    plt.hist(ages, bins=range(0, 110, 5), color='purple', alpha=0.7, edgecolor='black', rwidth=0.85)
    plt.xlabel("Âge")
    plt.ylabel("Effectif")
    plt.title(f"Pyramide des âges au round {round_num}")
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    plt.show()
    plt.close()  # OPTIMISATION : libère la figure

In [ ]:
def graph_blessures(population, round_num):
    """
    Affiche l'état de santé moyen basé sur le % de PV restants.
    """
    etats = []
    for h in population:
        pv_max_total = sum(m['pv_max'] for m in h.corps.values())
        pv_actuel_total = sum(m['pv_actuels'] for m in h.corps.values())
        pourcentage = (pv_actuel_total / pv_max_total) * 100 if pv_max_total > 0 else 0
        etats.append(pourcentage)

    plt.figure(figsize=(8, 5))
    plt.hist(etats, bins=20, color='red', alpha=0.7, edgecolor='black')
    plt.xlabel("Pourcentage de vie restante (%)")
    plt.ylabel("Nombre de survivants")
    plt.title(f"État de santé de la population au round {round_num}")
    plt.axvline(x=50, color='black', linestyle='--', label='Seuil critique (50%)')
    plt.legend()
    plt.show()
    plt.close()  # OPTIMISATION : libère la figure

In [ ]:
def graph_force_vitesse(population, round_num):
    forces = [h.force for h in population]
    vitesses = [h.vitesse for h in population]

    plt.figure(figsize=(8, 6))
    plt.scatter(forces, vitesses, alpha=0.6, c='blue', edgecolors='w', linewidth=0.5)
    plt.xlabel("Force")
    plt.ylabel("Vitesse")
    plt.title(f"Profil de combat (Force vs Vitesse) au round {round_num}")
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.show()
    plt.close()  # OPTIMISATION : libère la figure

In [ ]:
def graph_etat_mental(population, round_num):
    mentaux = [h.mental for h in population]

    plt.figure(figsize=(8, 5))
    plt.hist(mentaux, bins=20, color='indigo', alpha=0.7, edgecolor='black')
    plt.xlabel("Santé Mentale (0 = Brisé, 1 = Stable)")
    plt.ylabel("Nombre d'individus")
    plt.title(f"État psychologique au round {round_num}")
    plt.axvline(x=0.3, color='red', linestyle='--', label='Seuil de panique')
    plt.legend()
    plt.show()
    plt.close()  # OPTIMISATION : libère la figure

In [ ]:
from collections import Counter

def graph_regions(population, round_num):
    # OPTIMISATION : un seul passage avec Counter au lieu de regions.count(r)
    # dans une boucle (qui était en O(n^2)).
    compte = Counter(h.region_key for h in population)
    unique_regions = list(compte.keys())
    counts = [compte[r] for r in unique_regions]

    plt.figure(figsize=(10, 5))
    plt.bar(unique_regions, counts, color='teal', edgecolor='black')
    plt.xlabel("Région d'origine")
    plt.ylabel("Nombre de survivants")
    plt.title(f"Survivants par région au round {round_num}")
    plt.xticks(rotation=45, ha='right')
    plt.tight_layout()
    plt.show()
    plt.close()  # OPTIMISATION : libère la figure

In [ ]:
def graph_richesse(population, round_num):
    richesses = [h.richesse for h in population]

    plt.figure(figsize=(8, 5))
    plt.hist(richesses, bins=50, color='gold', alpha=0.7, edgecolor='black')
    plt.xlabel("Richesse ($)")
    plt.ylabel("Nombre de survivants")
    plt.title(f"Distribution des richesses au round {round_num}")
    plt.yscale('log') # Souvent utile car les riches sont très riches
    plt.grid(axis='y', linestyle='--', alpha=0.5)
    plt.show()
    plt.close()  # OPTIMISATION : libère la figure

## 5 Tournoi avec moi

In [ ]:
def integrer_joueur(population, prenom, age, sexe, taille, poids, mental, niveau_entrainement, intelligence=100):
    """
    Crée un joueur personnalisé en définissant son physique (Taille/Poids).
    La Force et la Vitesse sont calculées automatiquement.
    """
    # BUGFIX : validation des entrees. Des valeurs <= 0 (taille/poids) produiraient
    # des PV et une force negatifs (combattant "invincible" ou degats qui soignent).
    if taille <= 0 or poids <= 0:
        raise ValueError("taille et poids doivent etre strictement positifs.")
    if age < 0:
        raise ValueError("age doit etre positif.")
    if not (0.0 <= mental <= 1.0):
        raise ValueError("mental doit etre dans [0, 1].")
    if not (0.0 <= niveau_entrainement <= 1.0):
        raise ValueError("niveau_entrainement doit etre dans [0, 1].")

    # 1. Création de la structure de base
    # OPTIMISATION : on alloue l'objet via __new__ sans passer par __init__, qui
    # générerait un humain aléatoire complet (âge, physique, santé, richesse...)
    # dont ~90% des champs seraient ensuite écrasés. Cela évite des dizaines de
    # tirages aléatoires inutiles et corrige au passage la région/richesse/classe
    # incohérentes qui étaient héritées du Human() aléatoire de base.
    joueur = Human.__new__(Human)
    Human._counter += 1
    joueur.id = Human._counter
    joueur.region_key = 'Joueur'
    joueur.richesse = 0.0
    joueur.classe_sociale = 'Joueur'

    # 2. Identité et Physique imposés
    joueur.prenom = prenom
    joueur.age = age
    joueur.sexe = sexe
    joueur.taille = taille
    joueur.poids = poids

    # On neutralise les stats parasites (maladie / grand âge) pour qu'un joueur
    # ne soit jamais déclaré incapable de se battre à tort.
    joueur.est_malade = False
    joueur.severite_maladie = 0.0
    joueur._handicap_grand_age = False

    # 3. Calcul automatique des Stats de combat (basé sur le modèle Human)
    # Force dépend du poids
    # BUGFIX : meme borne que dans _calculer_stats_combat (force >= 1).
    # 3.3 : meme modele de masse utile que la classe Human.
    imc_j = joueur.poids / (joueur.taille / 100) ** 2
    poids_seuil_j = 25 * (joueur.taille / 100) ** 2
    masse_utile_j = joueur.poids if imc_j <= 25 else poids_seuil_j + (joueur.poids - poids_seuil_j) * 0.2
    joueur.force = max(1.0, random.gauss(30, 10) + masse_utile_j * 0.4)

    # Vitesse dépend de l'âge
    if joueur.age < 25:
        facteur_age_vitesse = 1.0
    else:
        facteur_age_vitesse = max(0.5, 1 - (joueur.age - 25) * 0.01)
    joueur.vitesse = random.gauss(50, 8) * facteur_age_vitesse  # OPTIMISATION #1

    # Sauvegarde des maxs
    joueur.force_max = joueur.force
    joueur.vitesse_max = joueur.vitesse

    # 4. Mental et Compétences
    joueur.mental = mental
    joueur.mental_max = mental
    joueur.niveau_entrainement = niveau_entrainement
    joueur.resilience = max(0.05, min(1.0, 0.4 + 0.4 * niveau_entrainement + 0.2 * mental))  # 5.1
    joueur.intelligence = intelligence

    # Application du bonus d'entraînement (comme dans la classe Human)
    joueur.force += joueur.niveau_entrainement * 20
    joueur.vitesse += joueur.niveau_entrainement * 10
    joueur.force_max = joueur.force
    joueur.vitesse_max = joueur.vitesse

    # 5. Recalcul des PV du corps en fonction du NOUVEAU poids
    # (Sinon on aurait les PV d'un humain random)
    # OPTIMISATION #5 : meme construction factorisee que Human._initialiser_corps.
    joueur.corps = Human._construire_corps(joueur.poids)
    joueur.pv_total = sum(m['pv_actuels'] for m in joueur.corps.values())

    # Marqueur pour ne jamais écraser un joueur déjà inséré
    joueur._est_joueur = True

    # Vérification finale
    joueur._verifier_capacite_combat()

    # 6. Insertion aléatoire
    # BUGFIX : on évite d'écraser un emplacement déjà occupé par un autre joueur.
    # Avant, deux appels successifs pouvaient tirer le même index et le 2e joueur
    # écrasait le 1er -> un seul joueur réel dans le tournoi.
    candidats = [i for i, h in enumerate(population) if not getattr(h, '_est_joueur', False)]
    # BUGFIX : garde-fou si tous les emplacements sont déjà des joueurs
    # (random.choice([]) lèverait un IndexError peu explicite).
    if not candidats:
        raise ValueError("Aucun emplacement libre dans la population pour insérer un joueur.")
    index_aleatoire = random.choice(candidats)
    population[index_aleatoire] = joueur

    print(f">>> JOUEUR AJOUTÉ : {prenom} (ID:{joueur.id}) | {taille}cm, {poids}kg | For calculée:{joueur.force:.0f} | Vit calculée:{joueur.vitesse:.0f}")

    return joueur.id

In [ ]:
if __name__ == "__main__":
    TAILLE_POPULATION = 8192
    print(f"--- Génération d'une population ({TAILLE_POPULATION} Humains) ---")
    population = []

    for _ in range(TAILLE_POPULATION):
        h = Human()
        population.append(h)

    print(f"Population générée : {len(population)} individus.")

    # --- INTEGRATION DES 2 JOUEURS ---
    ids_joueurs = []

    # JOUEUR 1 : Un "Colosse" (Lourd, Grand -> Fort mais lent)
    ids_joueurs.append(integrer_joueur(
        population,
        prenom="Pierre",
        age=26,
        sexe="Homme",
        taille=190,       # Très grand
        poids=94,        # Très lourd -> Force énorme calculée
        mental=0.7,
        niveau_entrainement=0.5,
        intelligence=110
    ))

    # JOUEUR 2 : Un "Vif" (Léger, Petit -> Faible mais rapide)
    ids_joueurs.append(integrer_joueur(
        population,
        prenom="Alice",
        age=24,
        sexe="Femme",
        taille=157,       # Taille moyenne
        poids=49,         # Léger -> Force faible calculée, mais vitesse max
        mental=1.0,
        niveau_entrainement=0.7, # Très bon technique pour compenser la force
        intelligence=149
    ))

    print("\nLancement du tournoi Hunger Games...")

    # On passe la liste des IDs à surveiller
    vainqueur = hunger_games(population, tracked_ids=ids_joueurs)

    print(f"\nDétails du champion : {vainqueur}")